In [ ]:
# !pip install jieba
# !pip install evaluate

In [1]:
import jieba
import evaluate


c:\Users\HP\miniconda3\envs\pytorch_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# English Example
predictions = ["hello, I don't understand."]
references = [
    ["hello, I don't know."]
]
bleu = evaluate.load("bleu")
results = bleu.compute(predictions=predictions, references=references)
print(results)

In [ ]:
# Chinese Example
space_sent = "因 此 ， 我 们 知 道 ， 洪 水 是 气 候 和 气 候 变 化 的 结 果 ， 不 同 的 发 生 在 不 同 的 情 况 下 发 生 了 不 同 的 。"
sent = ''
for word in space_sent:
    if word != " ":
        sent += word
print(sent)
words = list(jieba.cut(sent, cut_all=False)) # tokeize sentence using jieba
sent_pred = ''
for word in words:
    if sent_pred == '':
        sent_pred += word
    else:
        sent_pred += ' ' + word
print(sent_pred)

print()

sent = "洪水的产生是气候和河道共同作用的结果，不同河道形态下洪水产生的特点是不同的。"
print(sent)
words = list(jieba.cut(sent, cut_all=False)) # tokeize sentence using jieba
sent_ref = ''
for word in words:
    if sent_ref == '':
        sent_ref += word
    else:
        sent_ref += ' ' + word
print(sent_ref)

print()

predictions = [sent_pred]
references = [
    [sent_ref]
]
bleu = evaluate.load("bleu")
results = bleu.compute(predictions=predictions, references=references)
print(results)

In [8]:
import os
import torch
import jieba
from tqdm import tqdm
from nltk import word_tokenize
from translator_en2cn import get_config, get_model, greedy_decode
from tokenization import PrepareData

# 1. 设置路径
model_path = 'save/models/best_model.pt'       # 模型路径
test_file = 'data/en-cn/test.txt'              # 测试集源文件
output_file = 'save/predictions_segmented.txt'   # 输出文件

# 2. 基础配置
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
config = get_config(debug=False)
UNK, PAD = 1, 0
print(f"当前设备: {device}")

当前设备: cuda


In [9]:
# 加载数据字典以获取词汇表大小
print("正在加载词典...")
data = PrepareData(config['train_file'], config['dev_file'], config['batch_size'], UNK, PAD)
src_vocab_size = len(data.en_word_dict)
tgt_vocab_size = len(data.cn_word_dict)
print(f"词典加载完成：源词汇 {src_vocab_size}, 目标词汇 {tgt_vocab_size}")

正在加载词典...
词典加载完成：源词汇 5493, 目标词汇 2519


In [10]:
# 初始化模型架构并加载保存的权重
print(f"正在加载模型: {model_path} ...")
model = get_model(config, src_vocab_size, tgt_vocab_size).to(device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()
print("✅ 模型就绪！")

正在加载模型: save/models/best_model.pt ...
✅ 模型就绪！


In [11]:
# 开始翻译并同步分词
print("开始翻译并同步分词测试集...")
if not os.path.exists('save'): os.makedirs('save')

with open(test_file, 'r', encoding='utf-8') as f_in, \
     open(output_file, 'w', encoding='utf-8') as f_out:
    
    lines = f_in.readlines()
    for line in tqdm(lines):
        line = line.strip()
        if not line: continue
        
        # 解析测试集 (EN \t ZH)
        parts = line.split('\t')
        en_text = parts[0]
        
        # 英文预处理
        en_tokens = ["BOS"] + word_tokenize(en_text.lower()) + ["EOS"]
        en_ids = [data.en_word_dict.get(w, UNK) for w in en_tokens]
        src = torch.tensor([en_ids]).to(device)
        src_mask = (src != PAD).unsqueeze(1).unsqueeze(1).to(device)
        
        # 翻译
        with torch.no_grad():
            model_out = greedy_decode(model, src, src_mask, data.cn_word_dict, config['seq_len'], device)
        
        # ID 转文字
        pred_sent = ""
        for i in range(1, model_out.size(0)):
            sym = data.cn_index_dict[model_out[i].item()]
            if sym == 'EOS': break
            pred_sent += sym
            
        # 中文分词
        words = jieba.cut(pred_sent, cut_all=False)
        segmented_sent = " ".join(words)
        
        # 保存
        f_out.write(segmented_sent + '\n')

print(f"\n✅ 处理完成！结果已保存至: {output_file}")

开始翻译并同步分词测试集...


100%|██████████| 1817/1817 [02:31<00:00, 12.02it/s]


✅ 处理完成！结果已保存至: save/predictions_segmented.txt


In [12]:
# Final BLEU evaluation on your own files 
import os
import jieba
import evaluate

# 1) Set file paths
# Prediction file: one predicted Chinese sentence per line.
# Reference file can be either:
#   - plain txt: one Chinese reference sentence per line
#   - tsv file: each line like "english\tchinese", and we will read the Chinese part
pred_file_path = "save/predictions_segmented.txt"   # 使用刚才生成的分词文件
ref_file_path = "data/en-cn/test.txt"          # 测试集路径

# If your reference file is TSV (en\tzh), set this to True.
# If your reference file is plain Chinese txt (one sentence per line), set this to False.
ref_is_tsv = True


def zh_tokenize_to_spaced(sentence: str) -> str:
    """Tokenize Chinese sentence with jieba and join tokens by spaces for BLEU."""
    sentence = sentence.strip()
    if not sentence:
        return ""
    tokens = list(jieba.cut(sentence, cut_all=False))
    return " ".join(tokens)


def read_predictions(path: str):
    """Read predicted Chinese sentences (one line = one sentence)."""
    preds = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            # 因为我们的文件已经分词过了，直接添加即可
            preds.append(line)
    return preds


def read_references(path: str, is_tsv: bool):
    """Read reference Chinese sentences.

    Returns format required by evaluate BLEU:
    references = [[ref1], [ref2], ...]
    """
    refs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            if is_tsv:
                parts = line.split("\t")
                if len(parts) < 2:
                    continue
                zh_sent = parts[1]
            else:
                zh_sent = line

            refs.append([zh_tokenize_to_spaced(zh_sent)])
    return refs


# 2) Sanity checks
if not os.path.exists(pred_file_path):
    raise FileNotFoundError(f"Prediction file not found: {pred_file_path}")
if not os.path.exists(ref_file_path):
    raise FileNotFoundError(f"Reference file not found: {ref_file_path}")

# 3) Batch preprocess
processed_preds = read_predictions(pred_file_path)
processed_refs = read_references(ref_file_path, ref_is_tsv)

if len(processed_preds) == 0:
    raise ValueError("Prediction file is empty after preprocessing.")
if len(processed_refs) == 0:
    raise ValueError("Reference file is empty after preprocessing.")

# Align lengths if they are not exactly the same.
n = min(len(processed_preds), len(processed_refs))
if len(processed_preds) != len(processed_refs):
    print(f"Warning: predictions({len(processed_preds)}) != references({len(processed_refs)}). Use first {n} pairs.")
processed_preds = processed_preds[:n]
processed_refs = processed_refs[:n]

# 4) Compute BLEU
bleu = evaluate.load("bleu")
results = bleu.compute(predictions=processed_preds, references=processed_refs)

print("最终测试集 BLEU 分数:", results)
print(f"有效评估句子数: {n}")

# Optional: print a few examples for report writing
show_k = min(5, n)
print("\n样例(分词后)：")
for i in range(show_k):
    print(f"[{i}] PRED: {processed_preds[i]}")
    print(f"[{i}] REF : {processed_refs[i][0]}")
    print("-")

最终测试集 BLEU 分数: {'bleu': 0.2121700495382972, 'precisions': [0.5699452211593492, 0.2603226425965047, 0.14900546702338024, 0.09586217051980563], 'brevity_penalty': 0.9888614516345505, 'length_ratio': 0.9889230271668823, 'translation_length': 12231, 'reference_length': 12368}
有效评估句子数: 1817

样例(分词后)：
[0] PRED: 他 不会 傻到 去 娶 她 。
[0] REF : 他 聪明 到 不会 娶 她 。
-
[1] PRED: 他 希望 他们 依然 就 没 成功 。
[1] REF : 他本 希望 可以 成功 ， 但是 他 没有 。
-
[2] PRED: 这 是 我 见 过 最好 的 电 影响 了 。
[2] REF : 这 是 我 看过 的 最 差劲 的 电影 了 。
-
[3] PRED: 她 在 洗澡 。
[3] REF : 她 在 浴室 。
-
[4] PRED: 人类 是 唯一 能够 彼此 交 动物 的 动物 。
[4] REF : 人 是 唯一 会 使用 火 的 动物 。
-


In [ ]:
# ============================================================
# 实验C: Cosine Warmup (peak_lr=2e-4, warmup=3) 测试集 BLEU
# ============================================================
import os
import torch
import jieba
import evaluate
from tqdm import tqdm
from nltk import word_tokenize
from translator_en2cn_cosine import get_config, get_model, greedy_decode
from tokenization import PrepareData

EXPERIMENT_NAME = 'cosine_warmup_lr2e-4_warm3'
model_path = f'save/models-cosine/{EXPERIMENT_NAME}/best_model.pt'
test_file = 'data/en-cn/test.txt'
output_file = f'save/predictions_segmented_{EXPERIMENT_NAME}.txt'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
config = get_config(debug=False)
UNK, PAD = 1, 0
print(f"当前设备: {device}")
print(f"实验: {EXPERIMENT_NAME} | peak_lr={config.get('peak_lr', 'N/A')} | warmup={config.get('warmup_epochs', 'N/A')}")

print("正在加载词典...")
data = PrepareData(config['train_file'], config['dev_file'], config['batch_size'], UNK, PAD)
print(f"词典加载完成：源词汇 {len(data.en_word_dict)}, 目标词汇 {len(data.cn_word_dict)}")

print(f"正在加载实验C模型: {model_path} ...")
model = get_model(config, len(data.en_word_dict), len(data.cn_word_dict)).to(device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()
print("✅ 实验C模型就绪！")

# ---- 翻译测试集 ----
print("开始翻译测试集...")
if not os.path.exists('save'): os.makedirs('save')

with open(test_file, 'r', encoding='utf-8') as f_in, \
     open(output_file, 'w', encoding='utf-8') as f_out:
    lines = f_in.readlines()
    for line in tqdm(lines):
        line = line.strip()
        if not line: continue
        parts = line.split('\t')
        en_text = parts[0]
        en_tokens = ["BOS"] + word_tokenize(en_text.lower()) + ["EOS"]
        en_ids = [data.en_word_dict.get(w, UNK) for w in en_tokens]
        src = torch.tensor([en_ids]).to(device)
        src_mask = (src != PAD).unsqueeze(1).unsqueeze(1).to(device)
        with torch.no_grad():
            model_out = greedy_decode(model, src, src_mask, data.cn_word_dict, config['seq_len'], device)
        pred_sent = ""
        for j in range(1, model_out.size(0)):
            sym = data.cn_index_dict[model_out[j].item()]
            if sym == 'EOS': break
            pred_sent += sym
        words = jieba.cut(pred_sent, cut_all=False)
        f_out.write(" ".join(words) + '\n')

print(f"\n✅ 翻译完成 -> {output_file}")

# ---- BLEU 评估 ----
def zh_tokenize(s):
    s = s.strip()
    return " ".join(jieba.cut(s, cut_all=False)) if s else ""

def read_preds(path):
    preds = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip(): preds.append(line.strip())
    return preds

def read_refs(path):
    refs = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            parts = line.split('\t')
            if len(parts) < 2: continue
            refs.append([zh_tokenize(parts[1])])
    return refs

preds = read_preds(output_file)
refs = read_refs(test_file)
n = min(len(preds), len(refs))

bleu = evaluate.load("bleu")
results = bleu.compute(predictions=preds[:n], references=refs[:n])

print("=" * 70)
print(f"🔥 实验C ({EXPERIMENT_NAME}) 测试集 BLEU:")
print(results)
print(f"评估句子数: {n}")
print("=" * 70)

show_k = min(5, n)
print("\n翻译示例(分词后)：")
for i in range(show_k):
    print(f"[{i}] PRED: {preds[i]}")
    print(f"[{i}] REF : {refs[i][0]}")
    print("-")

# 与基线对比
print(f"\n📊 基线 BLEU: 0.2121 | 实验C BLEU: {results['bleu']:.4f} | 提升: {(results['bleu'] - 0.2121) * 100:.2f}%")

当前设备: cuda
实验: cosine_warmup_lr2e-4_warm3 | peak_lr=0.0002 | warmup=3
正在加载词典...
词典加载完成：源词汇 5493, 目标词汇 2519
正在加载实验C模型: save/models-cosine/cosine_warmup_lr2e-4_warm3/best_model.pt ...
✅ 实验C模型就绪！
开始翻译测试集...


 15%|█▌        | 275/1817 [00:52<02:28, 10.41it/s]

In [2]:
# ============================================================
# 实验B: Cosine Warmup (peak_lr=3e-4, warmup=5) 测试集 BLEU
# ============================================================
import os
import torch
import jieba
import evaluate
from tqdm import tqdm
from nltk import word_tokenize
from translator_en2cn_cosine import get_config, get_model, greedy_decode
from tokenization import PrepareData

EXPERIMENT_NAME = 'cosine_warmup_lr3e-4_warm5'
model_path = f'save/models-cosine/{EXPERIMENT_NAME}/best_model.pt'
test_file = 'data/en-cn/test.txt'
output_file = f'save/predictions_segmented_{EXPERIMENT_NAME}.txt'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
config = get_config(debug=False)
UNK, PAD = 1, 0
print(f"当前设备: {device}")
print(f"实验: {EXPERIMENT_NAME} | peak_lr={config.get('peak_lr', 'N/A')} | warmup={config.get('warmup_epochs', 'N/A')}")

print("正在加载词典...")
data = PrepareData(config['train_file'], config['dev_file'], config['batch_size'], UNK, PAD)
print(f"词典加载完成：源词汇 {len(data.en_word_dict)}, 目标词汇 {len(data.cn_word_dict)}")

print(f"正在加载实验B模型: {model_path} ...")
model = get_model(config, len(data.en_word_dict), len(data.cn_word_dict)).to(device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()
print("✅ 实验B模型就绪！")

# ---- 翻译测试集 ----
print("开始翻译测试集...")
if not os.path.exists('save'): os.makedirs('save')

with open(test_file, 'r', encoding='utf-8') as f_in, \
     open(output_file, 'w', encoding='utf-8') as f_out:
    lines = f_in.readlines()
    for line in tqdm(lines):
        line = line.strip()
        if not line: continue
        parts = line.split('\t')
        en_text = parts[0]
        en_tokens = ["BOS"] + word_tokenize(en_text.lower()) + ["EOS"]
        en_ids = [data.en_word_dict.get(w, UNK) for w in en_tokens]
        src = torch.tensor([en_ids]).to(device)
        src_mask = (src != PAD).unsqueeze(1).unsqueeze(1).to(device)
        with torch.no_grad():
            model_out = greedy_decode(model, src, src_mask, data.cn_word_dict, config['seq_len'], device)
        pred_sent = ""
        for j in range(1, model_out.size(0)):
            sym = data.cn_index_dict[model_out[j].item()]
            if sym == 'EOS': break
            pred_sent += sym
        words = jieba.cut(pred_sent, cut_all=False)
        f_out.write(" ".join(words) + '\n')

print(f"\n✅ 翻译完成 -> {output_file}")

# ---- BLEU 评估 ----
def zh_tokenize(s):
    s = s.strip()
    return " ".join(jieba.cut(s, cut_all=False)) if s else ""

def read_preds(path):
    preds = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip(): preds.append(line.strip())
    return preds

def read_refs(path):
    refs = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            parts = line.split('\t')
            if len(parts) < 2: continue
            refs.append([zh_tokenize(parts[1])])
    return refs

preds = read_preds(output_file)
refs = read_refs(test_file)
n = min(len(preds), len(refs))

bleu = evaluate.load("bleu")
results = bleu.compute(predictions=preds[:n], references=refs[:n])

print("=" * 70)
print(f"🏆 实验B ({EXPERIMENT_NAME}) 测试集 BLEU:")
print(results)
print(f"评估句子数: {n}")
print("=" * 70)

show_k = min(5, n)
print("\n翻译示例(分词后)：")
for i in range(show_k):
    print(f"[{i}] PRED: {preds[i]}")
    print(f"[{i}] REF : {refs[i][0]}")
    print("-")

# 与基线和实验C对比
print(f"\n📊 基线 BLEU: 0.2121 | 实验C BLEU: 0.2268 | 实验B BLEU: {results['bleu']:.4f}")
print(f"实验B vs 基线提升: {(results['bleu'] - 0.2121) * 100:.2f}%")
print(f"实验B vs 实验C提升: {(results['bleu'] - 0.2268) * 100:.2f}%")

当前设备: cuda
实验: cosine_warmup_lr3e-4_warm5 | peak_lr=0.0002 | warmup=3
正在加载词典...
词典加载完成：源词汇 5493, 目标词汇 2519
正在加载实验B模型: save/models-cosine/cosine_warmup_lr3e-4_warm5/best_model.pt ...
✅ 实验B模型就绪！
开始翻译测试集...


100%|██████████| 1817/1817 [03:28<00:00,  8.73it/s]



✅ 翻译完成 -> save/predictions_segmented_cosine_warmup_lr3e-4_warm5.txt
🏆 实验B (cosine_warmup_lr3e-4_warm5) 测试集 BLEU:
{'bleu': 0.23225147997473258, 'precisions': [0.596206867741666, 0.2895981087470449, 0.17324535092981402, 0.11114513165952235], 'brevity_penalty': 0.967213406815719, 'length_ratio': 0.9677393272962483, 'translation_length': 11969, 'reference_length': 12368}
评估句子数: 1817

翻译示例(分词后)：
[0] PRED: 他 不会 傻到 去 娶 她 。
[0] REF : 他 聪明 到 不会 娶 她 。
-
[1] PRED: 他 希望 所有 的 成功 ， 但 他 没有 办法 。
[1] REF : 他本 希望 可以 成功 ， 但是 他 没有 。
-
[2] PRED: 这 是 我 见 过 最 糟糕 的 旅行 。
[2] REF : 这 是 我 看过 的 最 差劲 的 电影 了 。
-
[3] PRED: 她 在 洗澡 。
[3] REF : 她 在 浴室 。
-
[4] PRED: 人类 是 唯一 能够 彼此 交谈 的 火灾 。
[4] REF : 人 是 唯一 会 使用 火 的 动物 。
-

📊 基线 BLEU: 0.2121 | 实验C BLEU: 0.2268 | 实验B BLEU: 0.2323
实验B vs 基线提升: 2.02%
实验B vs 实验C提升: 0.55%
